# Tr0ub4dor&3 or correct horse battery staple?

Guessing a password is a search whose cost is *alphabet size to the power of the
length*. Length is the exponent, and that is where the leverage is.

In [1]:
def seconds_to_crack(alphabet, length, guesses_per_second=1e10):
    return alphabet ** length / guesses_per_second

def human(s):
    for unit, size in [("years", 3.156e7), ("days", 86400), ("hours", 3600), ("minutes", 60)]:
        if s >= size:
            return f"{s / size:,.0f} {unit}"
    return f"{s:.0f} seconds"

cases = [("8 lowercase letters", 26, 8), ("8 chars from 95 symbols", 95, 8),
         ("4 words from a 2048-word list", 2048, 4), ("12 lowercase letters", 26, 12),
         ("12 chars from 95 symbols", 95, 12), ("6 words from a 2048-word list", 2048, 6)]
for label, A, L in cases:
    print(f"{label:32} {human(seconds_to_crack(A, L)):>20}")

8 lowercase letters                        21 seconds
8 chars from 95 symbols                        8 days
4 words from a 2048-word list              29 minutes
12 lowercase letters                         110 days
12 chars from 95 symbols              1,712,168 years
6 words from a 2048-word list               234 years


## Actually brute-forcing a (tiny) password

Time the search for 3, 4 and 5 lowercase letters and watch each extra letter
multiply the time by 26.

In [2]:
import itertools, string, time

def brute_force(target):
    alphabet = string.ascii_lowercase
    for guess in itertools.product(alphabet, repeat=len(target)):
        if "".join(guess) == target:
            return

for target in ["zzz", "zzzz", "zzzzz"]:            # the worst case: last guess
    t0 = time.perf_counter(); brute_force(target); dt = time.perf_counter() - t0
    print(f"{len(target)} letters: {26**len(target):>10,} guesses  {dt:8.3f} s")

3 letters:     17,576 guesses     0.001 s
4 letters:    456,976 guesses     0.030 s


5 letters: 11,881,376 guesses     0.852 s


## Hardware versus length

A 100× faster attacker shifts every bar by a constant factor; one more word
multiplies the work by 2 048.

In [3]:
print("4 words, attacker 100x faster:", human(seconds_to_crack(2048, 4, 1e12)))
print("5 words, same attacker:       ", human(seconds_to_crack(2048, 5, 1e12)))

4 words, attacker 100x faster: 18 seconds
5 words, same attacker:        10 hours


**What we used:** exponential versus polynomial growth, and why a constant factor
cannot compete with an exponent (session 8, sections 3, 8 and 9).